<a href="https://colab.research.google.com/github/bh1rg1v/colab/blob/main/Torrent_To_Google_Drive_Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Torrent To Google Drive Downloader

**Important Note:** To get more disk space:
> Go to Runtime -> Change Runtime and give GPU as the Hardware Accelerator.  You will get around 384GB to download any torrent you want.

### Install libtorrent and Initialize Session

In [ ]:
!apt install python3-libtorrent

import libtorrent as lt

ses = lt.session()
ses.listen_on(6881, 6891)
downloads = []

### Mount Google Drive
To stream files we need to mount Google Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

### Add From Torrent File
You can run this cell to add more files as many times as you want

In [ ]:
from google.colab import files

source = files.upload()
params = {
    "save_path": "/content/drive/My Drive/Torrent",
    "ti": lt.torrent_info(list(source.keys())[0]),
}
downloads.append(ses.add_torrent(params))

### Add From Magnet Link
You can run this cell to add more files as many times as you want

In [ ]:
params = {"save_path": "/content/drive/My Drive/Torrent"}

while True:
    magnet_link = input("Enter Magnet Link Or Type Exit: ")
    if magnet_link.lower() == "exit":
        break
    downloads.append(
        lt.add_magnet_uri(ses, magnet_link, params)
    )


### Start Download
Source: https://stackoverflow.com/a/5494823/7957705 and [#3 issue](https://github.com/FKLC/Torrent-To-Google-Drive-Downloader/issues/3) which refers to this [stackoverflow question](https://stackoverflow.com/a/6053350/7957705)

In [6]:
import time
from IPython.display import display
import ipywidgets as widgets

state_str = [
    "queued",
    "checking",
    "downloading metadata",
    "downloading",
    "finished",
    "seeding",
    "allocating",
    "checking fastresume",
]

layout = widgets.Layout(width="auto")
style = {"description_width": "initial"}
download_bars = [
    widgets.FloatSlider(
        step=0.01, disabled=True, layout=layout, style=style
    )
    for _ in downloads
]
display(*download_bars)

while downloads:
    next_shift = 0
    for index, download in enumerate(downloads[:]):
        bar = download_bars[index + next_shift]
        if not download.is_seed():
            s = download.status()

            bar.description = " ".join(
                [
                    download.name(),
                    str(s.download_rate / 1000),
                    "kB/s",
                    state_str[s.state],
                ]
            )
            bar.value = s.progress * 100
        else:
            next_shift -= 1
            ses.remove_torrent(download)
            downloads.remove(download)
            bar.close() # Seems to be not working in Colab (see https://github.com/googlecolab/colabtools/issues/726#issue-486731758)
            download_bars.remove(bar)
            print(download.name(), "complete")
    time.sleep(1)


FloatSlider(value=0.0, disabled=True, layout=Layout(width='auto'), step=0.01, style=SliderStyle(description_wi…

/tmp/ipykernel_544/3737971044.py:30: DeprecationWarning: is_seed() is deprecated
  if not download.is_seed():
/tmp/ipykernel_544/3737971044.py:35: DeprecationWarning: name() is deprecated
  download.name(),
/tmp/ipykernel_544/3737971044.py:30: DeprecationWarning: is_seed() is deprecated
  if not download.is_seed():
/tmp/ipykernel_544/3737971044.py:35: DeprecationWarning: name() is deprecated
  download.name(),
/tmp/ipykernel_544/3737971044.py:35: DeprecationWarning: name() is deprecated
  download.name(),
/tmp/ipykernel_544/3737971044.py:30: DeprecationWarning: is_seed() is deprecated
  if not download.is_seed():


KeyboardInterrupt: 

In [9]:
import time
import sys
import libtorrent as lt

# Configure your batch start here
start_file_num = 1  # Change this number for subsequent batches
max_batch_size_bytes = 200 * 1024 * 1024 * 1024  # 200 GB limit

# Get the active torrent handle from downloads list
if not downloads:
    raise ValueError("No active torrents found in 'downloads' list. Please add a torrent first.")

handle = downloads[0]
torrent_info = handle.get_torrent_info()
num_files = torrent_info.num_files()
files_storage = torrent_info.files()

# 1. Read all files, sort by name, and associate each with an index starting from 1
files_list = []
for i in range(num_files):
    file_path = files_storage.file_path(i)
    file_size = files_storage.file_size(i)
    files_list.append({"original_index": i, "path": file_path, "size": file_size})

# Sort alphabetically by path
files_list.sort(key=lambda x: x["path"])

# Associate with 1-based index
for rank, item in enumerate(files_list, start=1):
    item["rank"] = rank

print(f"Total files in torrent: {len(files_list)}")

# 2. Set all file priorities to 0 (don't download)
handle.prioritize_files([0] * num_files)

# 3. Select files starting from `start_file_num` up to 200GB
batch_files = []
current_batch_size = 0
next_start_num = None

for item in files_list:
    if item["rank"] >= start_file_num:
        if current_batch_size + item["size"] <= max_batch_size_bytes:
            batch_files.append(item)
            current_batch_size += item["size"]
        else:
            next_start_num = item["rank"]
            break

if not batch_files:
    print("No files remaining or the next single file exceeds the remaining limit.")
else:
    print(f"\nPreparing to download {len(batch_files)} files (Approx. {current_batch_size / (1024**3):.2f} GB):")
    file_priorities = list(handle.get_file_priorities())

    for item in batch_files:
        print(f"Rank #{item['rank']} | Size: {item['size'] / (1024**2):.2f} MB | {item['path']}")
        # Set target file priority to maximum (7)
        file_priorities[item["original_index"]] = 7

    handle.prioritize_files(file_priorities)

    # 4. Monitor download progress for the selected batch
    print("\nDownloading files...")
    while True:
        status = handle.status()
        # Check if files in this batch are completed
        file_progress = handle.file_progress()
        all_completed = True
        for item in batch_files:
            orig_idx = item["original_index"]
            target_size = item["size"]
            downloaded = file_progress[orig_idx]
            if downloaded < target_size:
                all_completed = False
                break

        print(f"\rProgress: {status.progress * 100:.2f}% | Download Speed: {status.download_rate / 1000:.1f} kB/s", end="")
        if all_completed:
            print("\nAll files in this batch are successfully downloaded and synced!")
            break
        time.sleep(5)

if next_start_num:
    print(f"\n>>> RUN THE NEXT BATCH STARTING WITH FILE NUMBER: {next_start_num} <<<")
else:
    print("\n>>> All torrent files have been successfully processed! <<<")

/tmp/ipykernel_544/2269901051.py:14: DeprecationWarning: get_torrent_info() is deprecated
  torrent_info = handle.get_torrent_info()


Total files in torrent: 464

Preparing to download 124 files (Approx. 199.72 GB):
Rank #1 | Size: 0.14 MB | reddit/comments/RC_2005-12.zst
Rank #2 | Size: 0.39 MB | reddit/comments/RC_2006-01.zst
Rank #3 | Size: 0.97 MB | reddit/comments/RC_2006-02.zst
Rank #4 | Size: 1.32 MB | reddit/comments/RC_2006-03.zst
Rank #5 | Size: 2.01 MB | reddit/comments/RC_2006-04.zst
Rank #6 | Size: 2.72 MB | reddit/comments/RC_2006-05.zst
Rank #7 | Size: 2.91 MB | reddit/comments/RC_2006-06.zst
Rank #8 | Size: 3.59 MB | reddit/comments/RC_2006-07.zst
Rank #9 | Size: 4.92 MB | reddit/comments/RC_2006-08.zst
Rank #10 | Size: 4.96 MB | reddit/comments/RC_2006-09.zst
Rank #11 | Size: 5.00 MB | reddit/comments/RC_2006-10.zst
Rank #12 | Size: 5.72 MB | reddit/comments/RC_2006-11.zst
Rank #13 | Size: 5.75 MB | reddit/comments/RC_2006-12.zst
Rank #14 | Size: 7.72 MB | reddit/comments/RC_2007-01.zst
Rank #15 | Size: 8.83 MB | reddit/comments/RC_2007-02.zst
Rank #16 | Size: 9.94 MB | reddit/comments/RC_2007-03.zst

KeyboardInterrupt: 

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
